# **UT/ST编写及验证**
在完成开源仓的算子开发后，本节将重点讲解算子的验证流程，主要包含UT、ST两种验证方式，核心内容如下：
- 环境准备
- UT验证
- ST验证
- 课后实践
---

## **1. 环境准备**
本文所有内容均存放于Sources文件夹。  
在开始创建算子工程前，先要对jupyter环境进行初始化。以下代码完成了初始化并将环境中的变量导入jupyter环境，并完成代码目录的创建。

In [ ]:
!mkdir -p Sources/06.04

import os, subprocess
env = subprocess.check_output("bash -l -c 'source $ASCEND_TOOLKIT_HOME/set_env.sh && env'", shell=True, text=True)
for line in env.splitlines():
    if "=" in line: os.environ.__setitem__(*line.split("=", 1))
print("\n🎉 Environment initialization process completed successfully!")

ops-math仓UT执行时依赖gawk，需要在执行前安装gawk，可执行以下命令安装：

In [ ]:
%%bash
set -e

GAWK_VERSION="5.3.0"
INSTALL_DIR="$HOME/.local"
RC_FILE="$HOME/.bashrc"  # 固定使用.bashrc，不存在则创建
PATH_LINE="export PATH=$INSTALL_DIR/bin:\$PATH"
# 确保.bashrc存在，且只添加一次PATH
touch "$RC_FILE" 
source "$RC_FILE"
if command -v gawk &> /dev/null; then
    echo "✅ gawk 已存在，版本：$(gawk --version | head -n1)"
    echo "✅ 跳过编译安装"
    exit 0
fi

[ ! -f "gawk-$GAWK_VERSION.tar.gz" ] && wget https://ftp.gnu.org/gnu/gawk/gawk-$GAWK_VERSION.tar.gz

tar -zxf gawk-$GAWK_VERSION.tar.gz --overwrite 
cd gawk-$GAWK_VERSION
./configure --prefix=$INSTALL_DIR
make -j$(nproc)
make install

grep -q "$INSTALL_DIR/bin" "$RC_FILE" || echo "$PATH_LINE" >> "$RC_FILE"
source "$RC_FILE"

echo "gawk 安装完成，路径：$(which gawk)"

In [ ]:
import os
os.environ["PATH"] = f"{os.environ['HOME']}/.local/bin:{os.environ['PATH']}"

## **2. UT验证**
UT（Unit Testing，简称 UT）是软件测试的一种，通常由开发者编写测试代码并运行。在ops-math算子仓中无需编译部署算子包，即可通过UT验证方式进行快速验证Infershape推导结果、Tiling实现逻辑、Kernel实现逻辑是否符合预期。  
UT目录结构如下，需用户手动创建：
```
${op_name}
...                                                     # 其他交付件
└── tests                                               # 测试交付件
    └── ut                                              # UT实现
        ├── op_host
        │   └── test_${op_name}_tiling.cpp              # Tiling UT实现
        │   └── test_${op_name}_infershape.cpp          # Infershape UT实现
        └── op_kernel
            └── test_${op_name}.cpp                     # Kernel UT实现
```

下面我们将基于之前课程开发的add算子，讲解UT验证。这里我们先准备好ops-math仓以及add算子工程。

In [ ]:
%%bash
# 清除可能存在的ops-math，以确保下载正常
rm -rf Sources/06.04/ops-math
# 下载ops-math开源仓
cd Sources/06.04;git clone https://gitcode.com/cann/ops-math.git;cd ../..

# 复制已准备好的add算子工程到开源仓experimental/math目录
cp -r src/add_custom Sources/06.04/ops-math/experimental/math

命令执行后，add_custom算子工程目录结构如下：

```
ops-math/
└── experimental/
    ├── conversion/
    ├── CMakeLists.txt
    ├── random/
    └── math/
        ├── add_custom/
        ├── CMakeLists.txt
        ├── op_graph/
        │   ├── add_custom_proto.h                       // 算子原型定义，用于图优化和融合阶段识别算子
        │   └── add_custom_graph_infer.cpp               // InferDataType文件，实现算子数据类型推导
        ├── op_host/
        │   ├── add_custom_def.cpp                       // 算子信息库，定义算子基本信息，如名称、输入输出、数据类型等
        │   ├── add_custom_tiling.cpp                    // Tiling实现
        │   └── add_custom_infershape.cpp                // InferShape实现
        ├── examples/
        │   └── test_aclnn_add_custom.cpp                // 算子aclnn调用代码
        └── op_kernel/
            ├── add_custom.h                             // 算子实现文件
            ├── add_custom.cpp                           // 算子核函数文件
            ├── add_custom_tiling_data.h                 // Tiling数据结构定义文件
            └── add_custom_tiling_key.h                  // TilingKey模板定义文件

```  

可执行以下代码查看目录结构:

In [ ]:
!cd Sources/06.04/ops-math/experimental/math/add_custom;find . -maxdepth 2 -print | sed -e 's;[^/]*/;|____;g;s;____|;    |;g'

### **2.1 Tiling UT**
Tiling UT用于验证host侧Tiling逻辑是否正确，在给定算子的输入后，Tiling能否正确执行、输出是否符合预期。代码存放于`${op_name}`/tests/ut/op_host目录下，文件名为test_`${op_name}`_tiling.cpp。以AddCustom算子为例，添加Tiling UT文件后，目录结构如下：

In [ ]:
# 创建UT存放文件夹，确保一定存在Tiling UT文件存放目录
!mkdir -p Sources/06.04/ops-math/experimental/math/add_custom/tests/ut/op_host
# 创建空UT文件
!touch Sources/06.04/ops-math/experimental/math/add_custom/tests/ut/op_host/test_add_custom_tiling.cpp
# 查看存在Tiling UT文件时整个算子工程目录结构
!cd Sources/06.04/ops-math/experimental/math/add_custom;find . -maxdepth 4 -print | sed -e 's;[^/]*/;|____;g;s;____|;    |;g'

#### **2.1.1 添加头文件**
Tiling UT文件需要统一包含iostream, gtest/gtest.h、tiling_context_faker.h、tiling_case_executor.h。这些是固定添加内容，如果有其他头文件，需要在这四个头文件基础上添加。

In [ ]:
%%writefile Sources/06.04/ops-math/experimental/math/add_custom/tests/ut/op_host/test_add_custom_tiling.cpp
#include <iostream>
#include <gtest/gtest.h>
#include "tiling_context_faker.h"
#include "tiling_case_executor.h"

#### **2.1.2 添加测试类**
Tiling UT文件需要定义一个测试类，继承自testing::Test。测试类建议`${OpName}`TilingTest，例如AddCustom算子的测试类为AddCustomTilingTest。继承Test类后，需要实现SetUpTestCase和TearDownTestCase静态方法，用于在所有测试用例执行前和执行后进行初始化和清理工作。

In [ ]:
%%writefile -a Sources/06.04/ops-math/experimental/math/add_custom/tests/ut/op_host/test_add_custom_tiling.cpp
class AddCustomTilingTest : public testing::Test {
protected:
    static void SetUpTestCase()
    {
        std::cout << "AddCustomTilingTest SetUp" << std::endl;
    }

    static void TearDownTestCase()
    {
        std::cout << "AddCustomTilingTest TearDown" << std::endl;
    }
};

#### **2.1.3 添加测试用例**
Tiling UT文件需要定义至少一个测试用例，通常使用TEST_F宏定义，格式为：
```c++
TEST_F(测试类, 用例名)
{
    // 1. 创建算子CompileInfo结构体
    struct AddCustomCompileInfo {
    } CompileInfo结构体;
    // 2. 调用TilingContextPara接口构造用例上下文。
  
    gert::TilingContextPara tilingContextPara(
    "算子名",
    {
        {{{输入1原始输入shape}, {输入1运行时shape}}, 输入1类型, 输入1排布格式},
        {{{输入2原始输入shape}, {输入2运行时shape}}, 输入2类型, 输入2排布格式},
    },
    {
        {{{输出1原始输入shape}, {输出1运行时shape}}, 输出1类型, 输出1排布格式},
    },
    {}, // 属性
    &CompileInfo结构体, // 编译信息，不需要传递额外的编译信息时可以为空结构体或nullptr
    8, // 设置可用核数, tiling实现内获取硬件产品信息核数时会获取该值
    262144, // 设置可用UB大小, tiling实现内获取硬件产品信息UB大小时会获取该值
    4096 // 设置tilingdata大小，该参数可不填
    );
 
    // 3. 设定预期结果。
    string expectTilingData = "tiling结构体第一个字段预期值 tiling结构体第二个字段预期值......";

    // 4.调用ExecuteTestCase接口执行用例。
    ExecuteTestCase(TilingContextPara对象, 预期状态, TilingKey, 预期TilingKey数据, 预期workspace);
```  

按照这个格式，我们对Add算子添加一个测试用例：

In [ ]:
%%writefile -a Sources/06.04/ops-math/experimental/math/add_custom/tests/ut/op_host/test_add_custom_tiling.cpp
TEST_F(AddCustomTilingTest, add_custom_tiling_test_1)
{
    struct AddCustomCompileInfo {
    } compileInfo;
    gert::TilingContextPara tilingContextPara(
        "AddCustom",
        {
            {{{8, 2048}, {8, 2048}}, ge::DT_FLOAT, ge::FORMAT_ND},
            {{{8, 2048}, {8, 2048}}, ge::DT_FLOAT, ge::FORMAT_ND},
        },
        {
            {{{8, 2048}, {8, 2048}}, ge::DT_FLOAT, ge::FORMAT_ND},
        },
        {
            /* attrs */
        },
        &compileInfo,
        64,
        262144
        ); 
    uint64_t expectTilingKey = 0;
    string expectTilingData = "16384 1 ";
    std::vector<size_t> expectWorkspaces = {0};
    ExecuteTestCase(tilingContextPara, ge::GRAPH_SUCCESS, expectTilingKey, expectTilingData, expectWorkspaces);
}

添加完用例后，我们可以通过执行以下命令来执行用例，通过用例是否通过判断Tiling实现逻辑是否正确：

In [ ]:
# 清理之前测试生成文件
!rm -rf build build_out
# 执行测试
!cd Sources/06.04/ops-math;bash build.sh -u --ophost --ops=add_custom --experimental

执行用例后，会有类似如下输出:
```shell
[----------] 1 test from AddCustomTilingTest
AddCustomTilingTest SetUp
[ RUN      ] AddCustomTilingTest.add_custom_tiling_test_1
[INFO] GE(18945,math_op_host_ut):2026-03-17-15:46:28.206.971 [op_context_builder_impl.cc:103]18945 CreateComputeNodeInfoImpl:Node AddCustom, compute_node_info attr_size 48, outputs_ins_info_size:48, offset:608, total_size:712.
[       OK ] AddCustomTilingTest.add_custom_tiling_test_1 (1 ms)
AddCustomTilingTest TearDown
[----------] 1 test from AddCustomTilingTest (1 ms total)
```

### **2.2 Kernel UT**
Kernel UT用于验证Device侧Kernel逻辑是否正确，在给定输入/Tiling参数后，Kernel能否正确执行、输出是否符合预期。代码存放于`${op_name}`/tests/ut/op_kernel目录下，文件名为test_`${op_name}`.cpp。以AddCustom算子为例，添加Kernel UT文件后，目录结构如下：

In [ ]:
# 创建UT存放文件夹，确保一定存在Kernel UT文件存放目录
!mkdir -p Sources/06.04/ops-math/experimental/math/add_custom/tests/ut/op_kernel
# 创建空UT文件
!touch Sources/06.04/ops-math/experimental/math/add_custom/tests/ut/op_kernel/test_add_custom.cpp
# 查看存在UT文件时整个算子工程目录结构
!cd Sources/06.04/ops-math/experimental/math/add_custom;find . -maxdepth 4 -print | sed -e 's;[^/]*/;|____;g;s;____|;    |;g'

#### **2.2.1 添加头文件**
Kernel UT文件需要统一包含gtest/gtest.h、tikicpulib.h、data_utils.h。这些是固定添加内容。另外需要添加核函数所在文件，直接引用op_kernel目录下的`${op_name}`.cpp文件即可。如果有其他头文件，需要在这四个文件基础上添加。


In [ ]:
%%writefile Sources/06.04/ops-math/experimental/math/add_custom/tests/ut/op_kernel/test_add_custom.cpp
#include <gtest/gtest.h>
#include "tikicpulib.h"
#include "data_utils.h"
#include "../../../op_kernel/add_custom.cpp"

#### **2.2.2 添加测试类**
Kernel UT文件需要定义一个测试类，继承自testing::Test。测试类建议`${OpName}`KernelTest，例如AddCustom算子的测试类为AddCustomKernelTest。继承Test类后，需要实现SetUpTestCase和TearDownTestCase静态方法，用于在所有测试用例执行前和执行后进行初始化和清理工作。  
不同于Infershape UT和Tiling UT，Kernel UT会输出大量计算结果，需对这些结果进行合规性验证以确保符合预期。为此，通常通过Python脚本实现测试数据生成、预期结果定义及结果正确性校验。我们建议在实现SetUpTestCase/TearDownTestCase统一做数据准备与清理工作（如拷贝数据目录、chmod、生成bin）。


In [ ]:
%%writefile -a Sources/06.04/ops-math/experimental/math/add_custom/tests/ut/op_kernel/test_add_custom.cpp
class AddCustomKernelTest : public testing::Test {
protected:
    static void SetUpTestCase()
    {
        std::cout << "AddCustomKernelTest SetUp\n" << std::endl;
        const std::string cmd = "cp -rf " + dataPath + " ./";
        system(cmd.c_str());
        system("chmod -R 755 ./add_custom_data/");
    }
    static void TearDownTestCase()
    {
        std::cout << "AddCustomKernelTest TearDown\n" << std::endl;
    }
private:
    const static std::string rootPath;
    const static std::string dataPath;
};

#### **2.2.3 添加测试用例**
Kernel UT文件需要定义至少一个测试用例，本质上是通过CPU调试核函数来测试kernel实现逻辑是否符合预期，通常使用TEST_F宏定义，格式为：
```c++
TEST_F(测试类, 用例名)
{
    // 1.设定输入shape/format/dtype，必要时准备ValueDepend输入值
    // 2.申请输入/输出/workspace/tiling内存
    uint8_t* x = (uint8_t*)AscendC::GmAlloc(...);
    uint8_t* y = (uint8_t*)AscendC::GmAlloc(...);
    uint8_t* z = (uint8_t*)AscendC::GmAlloc(...);
    uint8_t* workspace = (uint8_t*)AscendC::GmAlloc(...);
    uint8_t* tiling = (uint8_t*)AscendC::GmAlloc(sizeof(${op_name}TilingData));

    // 3.准备tiling数据（手动构造或由tiling函数生成）
    auto* tilingData = reinterpret_cast<${op_name}TilingData*>(tiling);
    tilingData->... = ...;

    // 4.设置tiling key并执行kernel
    ICPU_SET_TILING_KEY(tilingKey);
    AscendC::SetKernelMode(KernelMode::AIV_MODE);
    ICPU_RUN_KF(${op_name}, blockDim, x, y, z, workspace, tiling);

    // 5.结果校验
    EXPECT_EQ(..., ...);

    // 6.释放资源
    AscendC::GmFree(x);
    AscendC::GmFree(y);
    AscendC::GmFree(z);
    AscendC::GmFree(workspace);
    AscendC::GmFree(tiling);
}
```  

按照这个格式，我们对AddCustom算子添加一个测试用例：

In [ ]:
%%writefile -a Sources/06.04/ops-math/experimental/math/add_custom/tests/ut/op_kernel/test_add_custom.cpp
// 设置python生成的测试数据和预期结果路径
const std::string AddCustomKernelTest::rootPath = "../../../../";
const std::string AddCustomKernelTest::dataPath = rootPath + "experimental/math/add_custom/tests/ut/op_kernel/add_custom_data";

TEST_F(AddCustomKernelTest, test_case_0)
{
    size_t xByteSize = 8 * 2048 * sizeof(float);
    size_t yByteSize = 8 * 2048 * sizeof(float);
    size_t zByteSize = 8 * 2048 * sizeof(float);
    size_t tiling_data_size = sizeof(AddCustomTilingData);
    uint32_t numBlocks = 64;
    
    // 调用python脚本生成测试输入和预期输出
    system("cd ./add_custom_data/ && python3 gen_data.py");
    std::string input1 = "./add_custom_data/input1.bin";
    std::string input2 = "./add_custom_data/input2.bin";
    
    // 读取输入文件作为测试输入
    uint8_t* x = (uint8_t*)AscendC::GmAlloc(xByteSize);
    ReadFile(input1, xByteSize, x, xByteSize);

    uint8_t* y = (uint8_t*)AscendC::GmAlloc(yByteSize);
    ReadFile(input2, yByteSize, y, xByteSize);

    uint8_t* z = (uint8_t*)AscendC::GmAlloc(zByteSize);
    uint8_t* workspace = (uint8_t*)AscendC::GmAlloc(32);
    uint8_t* tiling = (uint8_t*)AscendC::GmAlloc(tiling_data_size);

    char* path_ = get_current_dir_name();
    std::string path(path_);

    // 设置tilingdata数据
    AddCustomTilingData* tilingDatafromBin = reinterpret_cast<AddCustomTilingData*>(tiling);
    tilingDatafromBin->totalLength = 8 * 2048;
    tilingDatafromBin->tileNum = 1;

    // 设置TilingKey和使用核的类型
    ICPU_SET_TILING_KEY(0);
    AscendC::SetKernelMode(KernelMode::AIV_MODE);

    // 调用核函数并传入构造好的输入和tilingdata
    ICPU_RUN_KF(add_custom<0>,
        numBlocks,
        x,
        y,
        z,
        workspace,
        (uint8_t *)(tilingDatafromBin));
    
    // 指定算子运算结果保存文件
    std::string outFile = "./add_custom_data/output.bin";
    WriteFile(outFile, z, zByteSize);

    // 释放资源
    AscendC::GmFree(x);
    AscendC::GmFree(y);
    AscendC::GmFree(z);
    AscendC::GmFree(workspace);
    AscendC::GmFree(tiling);
    
    // 执行对比
    system("cd ./add_custom_data/ && python3 compare_data.py");
    free(path_);
}

#### **2.2.4 编写数据生成和结果对比脚本**
上文用例添加时，我们调用其中的gen_data.py脚本生成测试数据和预期数据，并使用compare.py脚本对比测试结果和预期结果是否一致。通常我们会在Kernel UT用例所在文件夹下创建${op_name}_data文件夹，用于存放测试数据生成脚本和预期数据对比脚本。以AddCustom算子为例，添加文件后，目录结构如下：

In [ ]:
# 创建UT存放文件夹，确保一定存在脚本存放目录
!mkdir -p Sources/06.04/ops-math/experimental/math/add_custom/tests/ut/op_kernel/add_custom_data
# 创建空数据生成文件
!touch Sources/06.04/ops-math/experimental/math/add_custom/tests/ut/op_kernel/add_custom_data/gen_data.py
# 创建空数据对比文件
!touch Sources/06.04/ops-math/experimental/math/add_custom/tests/ut/op_kernel/add_custom_data/compare.py
# 查看存在Tiling UT文件时整个算子工程目录结构
!cd Sources/06.04/ops-math/experimental/math/add_custom;find . -maxdepth 4 -print | sed -e 's;[^/]*/;|____;g;s;____|;    |;g'

根据不同的算子编写符合算子逻辑的测试数据生成脚本，以AddCustom算子为例，脚本内容如下：

In [ ]:
%%writefile Sources/06.04/ops-math/experimental/math/add_custom/tests/ut/op_kernel/add_custom_data/gen_data.py
import numpy as np
import os

def gen_golden_data_simple():
    # 指定要生成的测试数据取值范围、Shape、类型
    input_x = np.random.uniform(1, 100, [8, 2048]).astype(np.float32)
    input_y = np.random.uniform(1, 100, [8, 2048]).astype(np.float32)
    # 根据实际运算逻辑得到预期结果
    golden = input_x + input_y

    # 保存输入和预期输出到指定文件
    input_x.tofile("./input1.bin")
    input_y.tofile("./input2.bin")
    golden.tofile("./golden.bin")


if __name__ == "__main__":
    # 清理bin文件
    os.system("rm -rf *.bin")
    gen_golden_data_simple()

数据对比脚本通常是读取实际结果和预期结果，然后对比两者是否一致。以AddCustom算子为例，脚本内容如下：

In [ ]:
%%writefile Sources/06.04/ops-math/experimental/math/add_custom/tests/ut/op_kernel/add_custom_data/compare.py
import sys
import numpy as np
import glob
import os

curr_dir = os.path.dirname(os.path.realpath(__file__))

def compare_data(golden_file_lists, output_file_lists, d_type):
    np_dtype = d_type
    data_same = True
    for gold, out in zip(golden_file_lists, output_file_lists):
        # 根据输出类型读取文件，并进行对比
        tmp_out = np.fromfile(out, np_dtype)
        tmp_gold = np.fromfile(gold, np_dtype)
        diff_res = np.isclose(tmp_out, tmp_gold, 0, 0, True)
        diff_idx = np.where(diff_res != True)[0]
        if len(diff_idx) == 0:
            print("PASSED!")
        else:
            print("FAILED!")
            for idx in diff_idx[:5]:
                print(f"index: {idx}, output: {tmp_out[idx]}, golden: {tmp_gold[idx]}")
            data_same = False
    return data_same

def get_file_lists(dtype):
    # 获取预期输出和实际输出文件列表
    golden_file_lists = sorted(glob.glob(curr_dir + "/*golden*.bin"))
    output_file_lists = sorted(glob.glob(curr_dir + "/*output*.bin"))
    return golden_file_lists, output_file_lists

def process(d_type):
    golden_file_lists, output_file_lists = get_file_lists(d_type)
    result = compare_data(golden_file_lists, output_file_lists, d_type)
    print("compare result:", result)
    return result

if __name__ == '__main__':
    ret = process(np.float32)
    exit(0 if ret else 1)

#### **2.2.5 编译脚本编写**
与其他UT测试不同，Kernel UT需要增加编译脚本（op_name/tests/ut/op_kernel/CMakeLists.txt）指定tiling实现文件和kernel实现内使用的宏定义具体值，通常格式为:
```cmake

if (UT_TEST_ALL OR OP_KERNEL_UT)
    # 算子自己的tiling实现文件路径
    set(算子名_tiling_files
        ${CMAKE_CURRENT_SOURCE_DIR}/../../../op_host/add_custom_tiling.cpp
        )
    # 使用AddOpTestCase
    # param1：算子名称，以kernel方式命名
    # param2：soc版本，多个以分号分隔，例如："ascend910_95;ascend910b"
    # param3：自定义编译选项，一般填写测试的一种典型数据类型组合，不需要则传入空字符串，例如："-DDTYPE_X=float"，多个使用空格分隔，例如："-DDTYPE_X=float -DDTYPE_Y=float"
    # param4：该算子依赖的所有tiling源码文件
    AddOpTestCase(算子名 "ascend910b" "" "${算子名_tiling_files}")
endif()
```

我们课程中使用的AddCustom算子示例，由于算子实现内使用了DTYPE_X1、DTYPE_X2、DTYPE_Y宏定义，编译脚本为：

In [ ]:
%%writefile Sources/06.04/ops-math/experimental/math/add_custom/tests/ut/op_kernel/CMakeLists.txt

if (UT_TEST_ALL OR OP_KERNEL_UT)
    # 需要将Tiling依赖的文件添加到CMakeLists.txt中
    # set(elewise_common_tiling_files
    #         ${CANN_ROOT}/ops/built-in/op_tiling/runtime/elewise_tiling.cc
    #         )
    # 算子自己的tiling文件路径
    set(add_custom_tiling_files
        ${CMAKE_CURRENT_SOURCE_DIR}/../../../op_host/add_custom_tiling.cpp
        )
    # 使用AddOpTestCase
    # param1：算子名称，以kernel方式命名
    # param2：soc版本，多个以分号分隔，例如："ascend910_95;ascend910b"
    # param3：自定义编译选项，一般填写测试的一种典型数据类型组合，不需要则传入空字符串，例如："-DDTYPE_X=float"，多个使用空格分隔，例如："-DDTYPE_X=float -DDTYPE_Y=float"
    # param4：该算子依赖的所有tiling源码文件
    AddOpTestCase(add_custom "ascend910b" "-DDTYPE_X1=float -DDTYPE_X2=float -DDTYPE_Y=float" "${add_custom_tiling_files}")
endif()


运行以下命令执行Kernel UT测试：

In [ ]:
!cd Sources/06.04/ops-math;bash build.sh -u --opkernel --ops=add_custom --experimental

正常运行时会有如下输出：
```shell
[       OK ] AddCustomKernelTest.test_case_0 (1100 ms)
AddCustomKernelTest TearDown

[----------] 1 test from AddCustomKernelTest (1100 ms total)

[----------] Global test environment tear-down
Global Environment TearDown
[==========] 1 test from 1 test suite ran. (1147 ms total)
[  PASSED  ] 1 test.
```

### **2.3 Infershape UT (可选)**
Infershape UT用于验证host侧Infershape逻辑是否正确，在给定算子的输入后，Infershape能否正确执行、输出是否符合预期。代码存放于`${op_name}`/tests/ut/op_host目录下，文件名为test_`${op_name}`_infershape.cpp。以add_custom算子为例，添加Infershape UT文件后，目录结构如下：

In [ ]:
# 创建UT存放文件夹
!mkdir -p Sources/06.04/ops-math/experimental/math/add_custom/tests/ut/op_host
# 创建空UT文件
!touch Sources/06.04/ops-math/experimental/math/add_custom/tests/ut/op_host/test_add_custom_infershape.cpp
# 查看存在Infershape UT文件时整个算子工程目录结构
!cd Sources/06.04/ops-math/experimental/math/add_custom/;find . -maxdepth 4 -print | sed -e 's;[^/]*/;|____;g;s;____|;    |;g'

#### **2.3.1 添加头文件**
Infershape UT文件需要统一包含iostream, gtest/gtest.h、infershape_context_faker.h、infershape_case_executor.h。这些是固定添加内容，如果有其他头文件，需要在这四个头文件基础上添加。

In [ ]:
%%writefile Sources/06.04/ops-math/experimental/math/add_custom/tests/ut/op_host/test_add_custom_infershape.cpp
#include <iostream>
#include <gtest/gtest.h>
#include "infershape_context_faker.h"
#include "infershape_case_executor.h"

#### **2.3.2 添加测试类**
Infershape UT文件需要定义一个测试类，继承自testing::Test。测试类建议${OpName}InfershapeTest，例如add_custom算子的测试类为AddInfershapeTest。继承Test类后，需要实现SetUpTestCase和TearDownTestCase静态方法，用于在所有测试用例执行前和执行后进行初始化和清理工作。

In [ ]:
%%writefile -a Sources/06.04/ops-math/experimental/math/add_custom/tests/ut/op_host/test_add_custom_infershape.cpp
class AddCustomInfershapeTest : public testing::Test
{
protected:
    static void SetUpTestCase()
    {
        std::cout << "AddCustomInfershapeTest SetUp" << std::endl;
    }

    static void TearDownTestCase()
    {
        std::cout << "AddCustomInfershapeTest TearDown" << std::endl;
    }
};

#### **2.3.3 添加测试用例**
Infershape UT文件需要定义至少一个测试用例，通常使用TEST_F宏定义，格式为：
```c++
TEST_F(测试类, 用例名)
{
    // 1. 调用InfershapeContextPara接口构造用例上下文。
    gert::InfershapeContextPara infershapeContextPara(
    "算子名",
    {
        {{{输入1原始输入shape}, {输入1运行时shape}}, 输入1类型, 输入1排布格式},
        {{{输入2原始输入shape}, {输入2运行时shape}}, 输入2类型, 输入2排布格式},
    },
    {
        {{{输出1原始输入shape}, {输出1运行时shape}}, 输出1类型, 输出1排布格式},
    });

    // 2. 设定预期结果。
    std::vector<std::vector<int64_t>> expectOutputShape = { 预期输出1Shape, 预期输出2Shape, ...};
    // 3.调用ExecuteTestCase接口执行用例。
    ExecuteTestCase(InfershapeContextPara对象, 预期状态, 预期输出1运行时shape, 预期输出2运行时shape, ...);
}
```  

按照这个格式，我们对AddCustom算子添加一个测试用例：

In [ ]:
%%writefile -a Sources/06.04/ops-math/experimental/math/add_custom/tests/ut/op_host/test_add_custom_infershape.cpp
TEST_F(AddCustomInfershapeTest, add_custom_infershape_test1)
{
    gert::InfershapeContextPara infershapeContextPara(
        "AddCustom",
        {
            {{{8, 2048}, {8, 2048}}, ge::DT_FLOAT, ge::FORMAT_ND},
            {{{8, 2048}, {8, 2048}}, ge::DT_FLOAT, ge::FORMAT_ND},
        },
        {
            {{{8, 2048}, {8, 2048}}, ge::DT_FLOAT, ge::FORMAT_ND},
        });
    std::vector<std::vector<int64_t>> expectOutputShape = {
        {8, 2048},
    };
    ExecuteTestCase(infershapeContextPara, ge::GRAPH_SUCCESS, expectOutputShape);
}

添加完用例后，我们可以通过执行以下命令来执行用例，通过用例是否通过判断InferShape逻辑是否正确：

In [ ]:
# 清理之前测试生成文件
!rm -rf build build_out
# 执行测试
!cd Sources/06.04/ops-math;bash build.sh -u --ophost --ops=add_custom --experimental

执行用例后，会有如下输出:
```shell
[       OK ] AddCustomInfershapeTest.add_custom_infershape_test1 (0 ms)
AddCustomInfershapeTest TearDown
[----------] 1 test from AddCustomInfershapeTest (0 ms total)

[----------] Global test environment tear-down
Global Environment TearDown
[==========] 1 test from 1 test suite ran. (13 ms total)
[  PASSED  ] 1 test.
```

---
## **3. ST验证**
ST全称System Test（系统测试），ST将算子集成到完整的系统环境中，验证算子在真实业务场景、端到端流程下的功能、性能、稳定性是否符合要求，我们可以通过以下方式进行Ascend算子的端到端验证：  

<table style="float: left; border-collapse: collapse; margin: 0; background: #f9f9f9; font-size: 14px; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Arial, sans-serif;">
    <tr>
        <th style="border: 1px solid #ddd; padding: 12px; text-align: left; font-weight: bold; min-width: 180px;">调用方式</th>
        <th style="border: 1px solid #ddd; padding: 12px; text-align: left; font-weight: bold; min-width: 180px;">使用条件</th>
    </tr>
    <tr>
        <td style="border: 1px solid #ddd; padding: 10px; vertical-align: middle; text-align: left;">单算子API</td>
        <td style="border: 1px solid #ddd; padding: 10px; vertical-align: middle; text-align: left;">算子工程编译部署</td>
    </tr>
    <tr>
        <td style="border: 1px solid #ddd; padding: 10px; vertical-align: middle; text-align: left;">单算子模型执行</td>
        <td style="border: 1px solid #ddd; padding: 10px; vertical-align: middle; text-align: left;">算子入图开发，算子工程编译部署</td>
    </tr>
    <tr>
        <td style="border: 1px solid #ddd; padding: 10px; vertical-align: middle; text-align: left;">IR构图</td>
        <td style="border: 1px solid #ddd; padding: 10px; vertical-align: middle; text-align: left;">算子入图开发，算子工程编译部署</td>
    </tr>
    <tr>
        <td style="border: 1px solid #ddd; padding: 10px; vertical-align: middle; text-align: left;">Pytorch框架调用</td>
        <td style="border: 1px solid #ddd; padding: 10px; vertical-align: middle; text-align: left;">插件适配开发，算子工程编译部署</td>
    </tr>
    <tr>
        <td style="border: 1px solid #ddd; padding: 10px; vertical-align: middle; text-align: left;">TensorFlow框架调用</td>
        <td style="border: 1px solid #ddd; padding: 10px; vertical-align: middle; text-align: left;">插件适配开发，算子工程编译部署</td>
    </tr>
        <tr>
        <td style="border: 1px solid #ddd; padding: 10px; vertical-align: middle; text-align: left;">Pybind调用</td>
        <td style="border: 1px solid #ddd; padding: 10px; vertical-align: middle; text-align: left;">算子工程编译部署</td>
    </tr>
</table>  

<div style="clear: both;"></div>

实际上，之前[Ascend C算子调用介绍](../03_intermediate_vector_operator_development/03.03_acl_pybind_call.ipynb)章节的内容就是端到端调用的ST测试。
由于ST需要端到端测试算子的功能，因此我们需要提前部署算子才能进行ST测试。执行以下命令部署我们已准备好的add算子：


In [ ]:
%%bash
# 清除可能存在的ops-math，以确保下载正常
rm -rf Sources/06.04/ops-math
# 下载ops-math开源仓
cd Sources/06.04;git clone https://gitcode.com/cann/ops-math.git;cd ../..

# 复制已准备好的AddCustom算子工程到开源仓experimental/math目录
cp -r src/add_custom Sources/06.04/ops-math/experimental/math

# 编译指定算子
cd Sources/06.04/ops-math;bash build.sh --pkg --soc=ascend910b --vendor_name=custom --ops=add_custom --experimental

#部署算子包
./build_out/cann-ops-math-custom_linux-aarch64.run --install-path=${HOME}/

add算子的目录结构如下：
```
add_custom
|-- CMakeLists.txt
|-- examples
|   `-- test_aclnn_add_custom.cpp                   // add算子的端到端调用ST测试文件, 创建算子工程时会自动生成空文件
|-- op_graph 
|   |-- add_custom_graph_infer.cpp                  // add算子的图推理文件, 创建算子工程时会自动生成模板文件
|   `-- add_custom_proto.h                          // add算子的图推理文件, 创建算子工程时会自动生成模板文件
|-- op_host
|   |-- add_custom_def.cpp                          // add算子的原型定义文件, 创建算子工程时会自动生成模板文件
|   |-- add_custom_infershape.cpp                   // add算子的形状推理文件, 创建算子工程时会自动生成模板文件
|   `-- add_custom_tiling.cpp                       // add算子的tiling实现文件, 创建算子工程时会自动生成模板文件
`-- op_kernel
    |-- add_custom.cpp                              // add算子的核函数实现文件, 创建算子工程时会自动生成模板文件
    |-- add_custom.h                                // add算子的算法实现文件, 创建算子工程时会自动生成模板文件
    |-- add_custom_tiling_data.h                    // add算子的tiling结构体定义文件, 创建算子工程时会自动生成模板文件
    `-- add_custom_tiling_key.h                     // add算子的tiling key模板文件, 创建算子工程时会自动生成模板文件

```

执行以下代码查看已有的算子工程目录：

In [ ]:
%%bash
cd Sources/06.04/ops-math/experimental/math/add_custom
find . -maxdepth 2 -print | sed -e 's;[^/]*/;|____;g;s;____|;    |;g'

单算子API可以直接在应用程序中调用，大致过程为：

1. 使用第一段接口aclnnXxxGetWorkspaceSize计算本次API调用计算过程中需要多少的workspace内存

2. 获取到本次API计算需要的workspace大小后，按照workspaceSize大小申请Device侧内存

3. 调用第二段接口aclnnXxx，调用对应的单算子二进制执行计算  

完整调用流程：  
<img src="./images/single_operator_api_invocation_flow.png" alt="single_operator_api_invocation_flow" width="350px" />

具体写法参照[Ascend C算子调用介绍](../03_intermediate_vector_operator_development/03.03_acl_pybind_call.ipynb)，需要注意，ops-math仓的单算子API的调用代码在算子工程下存放前需要放在examples文件夹，并将调用代码文件命名为test_aclnn_${op_name}.cpp。这里以AddCustom算子为例添加单算子API调用代码:：

In [ ]:
%%writefile Sources/06.04/ops-math/experimental/math/add_custom/examples/test_aclnn_add_custom.cpp
#include <algorithm>
#include <cstdint>
#include <cstdio>
#include <vector>

#include "aclnn/aclnn_base.h"
#include "aclnn/acl_meta.h"
#include "acl/acl_rt.h"
#include "aclnn_add_custom.h"

#define CHECK_ACL(expr)                                                                                 \
    do {                                                                                                \
        auto __ret = (expr);                                                                            \
        int32_t __code = static_cast<int32_t>(__ret);                                                   \
        if (__code != 0) {                                                                              \
            fprintf(stderr, "[ERROR] %s failed at %s:%d, ret=%d\n", #expr, __FILE__, __LINE__, __code); \
        }                                                                                               \
    } while (0)
int32_t main(int32_t argc, char** argv)
{
    const int32_t deviceId = 0;
    aclrtStream stream = nullptr;
    CHECK_ACL(aclnnInit(nullptr));
    CHECK_ACL(aclrtSetDevice(deviceId));
    CHECK_ACL(aclrtCreateStream(&stream));
    const std::vector<int64_t> shape = {8, 2048};
    const int64_t elementCount = shape[0] * shape[1];
    const size_t bufferSize = elementCount * sizeof(float);

    void* input0DeviceMem = nullptr;
    CHECK_ACL(aclrtMalloc(&input0DeviceMem, bufferSize, ACL_MEM_MALLOC_HUGE_FIRST));
    aclTensor* input0 = aclCreateTensor(shape.data(), shape.size(), ACL_FLOAT, nullptr, 0, ACL_FORMAT_ND,
                                        shape.data(), shape.size(), input0DeviceMem);

    void* input1DeviceMem = nullptr;
    CHECK_ACL(aclrtMalloc(&input1DeviceMem, bufferSize, ACL_MEM_MALLOC_HUGE_FIRST));
    aclTensor* input1 = aclCreateTensor(shape.data(), shape.size(), ACL_FLOAT, nullptr, 0, ACL_FORMAT_ND,
                                        shape.data(), shape.size(), input1DeviceMem);

    void* output0DeviceMem = nullptr;
    CHECK_ACL(aclrtMalloc(&output0DeviceMem, bufferSize, ACL_MEM_MALLOC_HUGE_FIRST));
    aclTensor* output0 = aclCreateTensor(shape.data(), shape.size(), ACL_FLOAT, nullptr, 0, ACL_FORMAT_ND,
                                         shape.data(), shape.size(), output0DeviceMem);
    std::vector<float> input0HostData(elementCount, 1.0);
    std::vector<float> input1HostData(elementCount, 2.0);
    std::vector<float> output0HostData(elementCount, 0.0);
    std::vector<float> goldenData(elementCount, 3.0);

    CHECK_ACL(aclrtMemcpy(input0DeviceMem, bufferSize, input0HostData.data(),
                          bufferSize, ACL_MEMCPY_HOST_TO_DEVICE));
    CHECK_ACL(aclrtMemcpy(input1DeviceMem, bufferSize, input1HostData.data(),
                          bufferSize, ACL_MEMCPY_HOST_TO_DEVICE));
    uint64_t workspaceSize = 0;
    aclOpExecutor* executor = nullptr;
    CHECK_ACL(aclnnAddCustomGetWorkspaceSize(input0, input1, output0, &workspaceSize, &executor));
    void* workspaceDeviceMem = nullptr;
    if (workspaceSize > 0) {
        CHECK_ACL(aclrtMalloc(&workspaceDeviceMem, workspaceSize, ACL_MEM_MALLOC_HUGE_FIRST));
    }
    CHECK_ACL(aclnnAddCustom(workspaceDeviceMem, workspaceSize, executor, stream));
    CHECK_ACL(aclrtSynchronizeStream(stream));
    CHECK_ACL(aclrtMemcpy(output0HostData.data(), bufferSize, output0DeviceMem,
                          bufferSize, ACL_MEMCPY_DEVICE_TO_HOST));

    printf("result is:\n");
    const int64_t previewCount = std::min<int64_t>(elementCount, 10);
    for (int64_t i = 0; i < previewCount; i++) { printf("%.1f ", output0HostData[i]); }
    printf("\ntest %s\n", std::equal(output0HostData.begin(), output0HostData.end(), goldenData.begin()) ? "pass" : "failed");
    aclDestroyTensor(input0);
    aclDestroyTensor(input1);
    aclDestroyTensor(output0);
    CHECK_ACL(aclrtFree(input0DeviceMem));
    CHECK_ACL(aclrtFree(input1DeviceMem));
    CHECK_ACL(aclrtFree(output0DeviceMem));
    if (workspaceSize > 0) {
        CHECK_ACL(aclrtFree(workspaceDeviceMem));
    }
    CHECK_ACL(aclrtDestroyStream(stream));
    CHECK_ACL(aclrtResetDevice(deviceId));
    CHECK_ACL(aclnnFinalize());
    return 0;
}

添加代码后即可使用ops-math仓的build.sh脚本进行编译测试，完成端到端的单算子API调用测试，即ST测试。

In [ ]:
!cd Sources/06.04/ops-math;source ${HOME}/vendors/custom_math/bin/set_env.bash;bash build.sh --run_example add_custom eager cust --vendor_name=custom --experimental

命令执行后应有如下输出:
```shell
result is:
3.0 3.0 3.0 3.0 3.0 3.0 3.0 3.0 3.0 3.0 
test pass
run test_aclnn_add_custom, execute samples success
```

---
## **课后实践**
上文课程演示了如何实现进行UT和ST测试，请对AddCustom算子补充UT测试，要求完成InferShape UT、Tiling UT、Kernel UT三个用例，测试输入Shape为[2, 1024]，测试输入类型为float32。

### **准备AddCustom算子的UT测试环境**
复制已准备好的AddCustom算子工程代码到ops-math仓的experimental/math目录下。

In [ ]:
%%bash
# 清除可能存在的ops-math，以确保下载正常
rm -rf Sources/06.04/ops-math
# 下载ops-math开源仓
cd Sources/06.04;git clone https://gitcode.com/cann/ops-math.git;cd ../..

# 复制已准备好的add_custom算子工程到开源仓experimental/math目录
cp -r src/add_custom Sources/06.04/ops-math/experimental/math

### **InferShape UT编写(可选)**
根据要求完成AddCustom算子的InferShape UT用例编写。

In [ ]:
# 创建UT存放文件夹
!mkdir -p Sources/06.04/ops-math/experimental/math/add_custom/tests/ut/op_host
# 创建空UT文件
!touch Sources/06.04/ops-math/experimental/math/add_custom/tests/ut/op_host/test_add_custom_infershape.cpp
# 查看存在Infershape UT文件时整个算子工程目录结构
!cd Sources/06.04/ops-math/experimental/math;find . -maxdepth 4 -print | sed -e 's;[^/]*/;|____;g;s;____|;    |;g'

In [ ]:
%%writefile Sources/06.04/ops-math/experimental/math/add_custom/tests/ut/op_host/test_add_custom_infershape.cpp
#include <gtest/gtest.h>
#include <iostream>
#include "infershape_context_faker.h"
#include "infershape_case_executor.h"

class AddInfershapeTest : public testing::Test
{
protected:
    static void SetUpTestCase()
    {
        std::cout << "AddInfershapeTest SetUp" << std::endl;
    }

    static void TearDownTestCase()
    {
        std::cout << "AddInfershapeTest TearDown" << std::endl;
    }
};

TEST_F(AddInfershapeTest, add_infershape_test1)
{
  // 补充UT
}


### **Tiling UT编写**
根据要求完成add_custom算子的Tiling UT用例编写。


In [ ]:
# 创建UT存放文件夹，确保一定存在Tiling UT文件存放目录
!mkdir -p Sources/06.04/ops-math/experimental/math/add_custom/tests/ut/op_host
# 创建空UT文件
!touch Sources/06.04/ops-math/experimental/math/add_custom/tests/ut/op_host/test_add_custom_tiling.cpp
# 查看存在Tiling UT文件时整个算子工程目录结构
!cd Sources/06.04/ops-math/experimental/math/add_custom;find . -maxdepth 4 -print | sed -e 's;[^/]*/;|____;g;s;____|;    |;g'

In [ ]:
%%writefile Sources/06.04/ops-math/experimental/math/add_custom/tests/ut/op_host/test_add_custom_tiling.cpp
#include <iostream>
#include <gtest/gtest.h>
#include "tiling_context_faker.h"
#include "tiling_case_executor.h"

class AddCustomTilingTest : public testing::Test {
protected:
    static void SetUpTestCase()
    {
        std::cout << "AddCustomTilingTest SetUp" << std::endl;
    }

    static void TearDownTestCase()
    {
        std::cout << "AddCustomTilingTest TearDown" << std::endl;
    }
};

TEST_F(AddCustomTilingTest, add_custom_tiling_test_1)
{
    // 补充用例实现代码
}

### **Kernel UT编写**
根据要求完成AddCustom算子的Kernel UT用例编写。

In [ ]:
# 创建UT存放文件夹，确保一定存在Kernel UT文件存放目录
!mkdir -p Sources/06.04/ops-math/experimental/math/add_custom/tests/ut/op_kernel
# 创建空UT文件
!touch Sources/06.04/ops-math/experimental/math/add_custom/tests/ut/op_kernel/test_add_custom.cpp

# 创建UT存放文件夹，确保一定存在脚本存放目录
!mkdir -p Sources/06.04/ops-math/experimental/math/add_custom/tests/ut/op_kernel/add_custom_data
# 创建空数据生成文件
!touch Sources/06.04/ops-math/experimental/math/add_custom/tests/ut/op_kernel/add_custom_data/gen_data.py
# 创建空数据对比文件
!touch Sources/06.04/ops-math/experimental/math/add_custom/tests/ut/op_kernel/add_custom_data/compare_data.py

#### **kernel ut代码编写**
根据练习要求的测试规格，编写AddCustom算子的Kernel UT用例代码。

In [ ]:
%%writefile -a Sources/06.04/ops-math/experimental/math/add_custom/tests/ut/op_kernel/test_add_custom.cpp
#include <gtest/gtest.h>
#include "tikicpulib.h"
#include "data_utils.h"
#include "../../../op_kernel/add_custom.cpp"

class AddCustomKernelTest : public testing::Test {
protected:
    static void SetUpTestCase()
    {
        std::cout << "AddCustomKernelTest SetUp\n" << std::endl;
        std::cout << "is_finite_test SetUp" << std::endl;
        const std::string cmd = "cp -rf " + dataPath + " ./";
        system(cmd.c_str());
        system("chmod -R 755 ./add_custom_data/");
    }
    static void TearDownTestCase()
    {
        std::cout << "AddCustomKernelTest TearDown\n" << std::endl;
    }
private:
    const static std::string rootPath;
    const static std::string dataPath;
};

const std::string AddCustomKernelTest::rootPath = "../../../../";
const std::string AddCustomKernelTest::dataPath = rootPath + "experimental/math/add_custom/tests/ut/op_kernel/add_custom_data";

TEST_F(AddCustomKernelTest, test_case_0)
{
    // 补充用例
}

#### **数据生成脚本编写**
根据练习要求修改测试数据生成脚本

In [ ]:
%%writefile Sources/06.04/ops-math/experimental/math/add_custom/tests/ut/op_kernel/add_custom_data/gen_data.py
import numpy as np
import os

def gen_golden_data_simple():
    # 指定要生成的测试数据取值范围、Shape、类型
    input_x = np.random.uniform(1, 100, [8, 2048]).astype(np.float32)
    input_y = np.random.uniform(1, 100, [8, 2048]).astype(np.float32)
    # 根据实际运算逻辑得到预期结果
    golden = input_x + input_y

    # 保存输入和预期输出到指定文件
    input_x.tofile("./input1.bin")
    input_y.tofile("./input2.bin")
    golden.tofile("./golden.bin")


if __name__ == "__main__":
    # 清理bin文件
    os.system("rm -rf *.bin")
    gen_golden_data_simple()

#### **对比脚本编写**
对比脚本可不修改

In [ ]:
%%writefile Sources/06.04/ops-math/experimental/math/add_custom/tests/ut/op_kernel/add_custom_data/compare_data.py
import sys
import numpy as np
import glob
import os

curr_dir = os.path.dirname(os.path.realpath(__file__))

def compare_data(golden_file_lists, output_file_lists, d_type):
    np_dtype = d_type
    data_same = True
    for gold, out in zip(golden_file_lists, output_file_lists):
        # 根据输出类型读取文件，并进行对比
        tmp_out = np.fromfile(out, np_dtype)
        tmp_gold = np.fromfile(gold, np_dtype)
        diff_res = np.isclose(tmp_out, tmp_gold, 0, 0, True)
        diff_idx = np.where(diff_res != True)[0]
        if len(diff_idx) == 0:
            print("PASSED!")
        else:
            print("FAILED!")
            for idx in diff_idx[:5]:
                print(f"index: {idx}, output: {tmp_out[idx]}, golden: {tmp_gold[idx]}")
            data_same = False
    return data_same

def get_file_lists(dtype):
    # 获取预期输出和实际输出文件列表
    golden_file_lists = sorted(glob.glob(curr_dir + "/*golden*.bin"))
    output_file_lists = sorted(glob.glob(curr_dir + "/*output*.bin"))
    return golden_file_lists, output_file_lists

def process(d_type):
    golden_file_lists, output_file_lists = get_file_lists(d_type)
    result = compare_data(golden_file_lists, output_file_lists, d_type)
    print("compare result:", result)
    return result

if __name__ == '__main__':
    ret = process(np.float32)
    exit(0 if ret else 1)

#### **编译脚本**
编译脚本不修改

In [ ]:
%%writefile Sources/06.04/ops-math/experimental/math/add_custom/tests/ut/op_kernel/CMakeLists.txt

if (UT_TEST_ALL OR OP_KERNEL_UT)
    # 需要将Tiling依赖的文件添加到CMakeLists.txt中
    # set(elewise_common_tiling_files
    #         ${CANN_ROOT}/ops/built-in/op_tiling/runtime/elewise_tiling.cc
    #         )
    # 算子自己的tiling文件路径
    set(add_custom_tiling_files
        ${CMAKE_CURRENT_SOURCE_DIR}/../../../op_host/add_custom_tiling.cpp
        )
    # 使用AddOpTestCase
    # param1：算子名称，以kernel方式命名
    # param2：soc版本，多个以分号分隔，例如："ascend910_95;ascend910b"
    # param3：自定义编译选项，一般填写测试的一种典型数据类型组合，不需要则传入空字符串，例如："-DDTYPE_X=float"，多个使用空格分隔，例如："-DDTYPE_X=float -DDTYPE_Y=float"
    # param4：该算子依赖的所有tiling源码文件
    AddOpTestCase(add_custom "ascend910b" "-DDTYPE_X1=float -DDTYPE_X2=float -DDTYPE_Y=float" "${add_custom_tiling_files}")
endif()


#### **执行用例测试**
执行以下代码，依次完成Infershape UT、Tiling UT、 Kernel UT测试：

In [ ]:
%%bash
cd Sources/06.04/ops-math
# 清除UT临时文件
rm -rf build build_out
# 测试InferShape UT 、Tiling UT
bash build.sh -u --ophost --ops=add_custom --experimental

# 清除UT临时文件
rm -rf build build_out
# 测试Kernel UT
bash build.sh -u --opkernel --ops=add_custom --experimental


参考答案：
InferShape UT 用例参考答案：

In [ ]:
!cat ./answer/06.04_answer/tests/ut/op_host/test_add_custom_infershape.cpp

Tiling UT用例参考答案：

In [ ]:
!cat ./answer/06.04_answer/tests/ut/op_host/test_add_custom_tiling.cpp

Kernel UT用例参考答案：

In [ ]:
!cat ./answer/06.04_answer/tests/ut/op_kernel/test_add_custom.cpp

Kernel UT数据生成脚本参考答案：

In [ ]:
!cat ./answer/06.04_answer/tests/ut/op_kernel/add_custom_data/gen_data.py